1. In this problem, you need to implement a function which calculates the $GCD$ of two numbers $(a, b)$ using the **Extended Euclid's Algorithm**. *(3 points)*
* It returns three values of the form $(d, x, y)$ where $d = GCD(a, b)$ and $(x, y)$ are two integers such that $d = ax + by$. It is given that $a \geq b \geq 0$ and at least one of them is non-zero.
* Note that you are not allowed to remove any of the assertions from the template code. Doing so will result in a score of 0 for this problem, and this is applicable for all problems where the template code has some assertions. You should just implement your solution in the space provided. There is no need to change anything in the template.

Obviously, you can refer to slides or any other material on the internet, but you should implement the algorithm yourself. This will make you really think about the algorithm and related things, like whether there are any edge cases which you need to consider, etc. Only when you do this, will you truly remember the algorithm and take something away from this workshop. Otherwise, this will just become another thing that you think you know well which'll fade away from memory after a day or two.

In [3]:
def extended_gcd(a, b):
    assert a >= b and b >= 0 and a + b > 0

    # Keep track of the coefficients of a and b.
    old_r, r = a, b
    old_x, x = 1, 0
    old_y, y = 0, 1

    # Euclidean algorithm, while updating Bezout coefficients.
    while r != 0:
        q = old_r // r
        old_r, r = r, old_r - q * r
        old_x, x = x, old_x - q * x
        old_y, y = y, old_y - q * y

    d, x, y = old_r, old_x, old_y

    assert a % d == 0 and b % d == 0
    assert d == a * x + b * y
    return (d, x, y)

In [4]:
print(extended_gcd(8, 4))
print(extended_gcd(27, 15))

(4, 0, 1)
(3, -1, 2)


2. Now that you know a connection between $GCD$ and $LCM$, write an efficient algorithm for finding the least common multiple of two **positive** integers. *(1 point)*
* Your function should return the $LCM$ of $(a, b)$

In [5]:
def lcm(a, b):
    # LCM(a, b) * GCD(a, b) = a * b.
    # Divide first to reduce the size of the intermediate product.
    assert a > 0 and b > 0
    d, _, _ = extended_gcd(max(a, b), min(a, b))
    return (a // d) * b

In [7]:
assert lcm(9, 3) == 9
assert lcm(17, 37) == 629

3. Now, try to use extended Euclid's algorithm to solve Diophantine equations efficiently. *(3 points)*

Given three numbers $a > 0$, $b > 0$, and $c$, the algorithm should return some $x$ and $y$ such that $ax + by = c$. It is given that a solution to the above equation exists for given $a, b ,c$.
(Again, you are not allowed to remove any assertions from the template.)

In [8]:
def diophantine(a, b, c):
    # Find d, x0, y0 such that a*x0 + b*y0 = d.
    d, x0, y0 = extended_gcd(max(a, b), min(a, b))
    if a < b:
        x0, y0 = y0, x0

    # Since d divides c, scale the Bezout coefficients by c/d.
    assert c % d == 0
    k = c // d
    x, y = k * x0, k * y0

    assert c % d == 0
    assert a * x + b * y == c

    return (x, y)

In [9]:
print(diophantine(34, 51, 170))
print(diophantine(91, 117, 65))

(-10, 10)
(20, -15)


4. Now that you know how to use extended Euclid's algorithm for finding modular inverses, implement an efficient algorithm for dividing $b$ by $a$ modulo $n$. *(4 points)*

Given three integers $a$, $b$, and $n$, such that $GCD(a, n) = 1$ and $n > 1$, the algorithm should return an integer $x$ such that

$0 \leq x \leq n-1$, and

$b  /  a = x \pmod{n}$ ( that is, $b = ax \pmod{n}$ ).
In particular, your task is to calculate the multiplicative inverse of $a \pmod{n}$.

In [10]:
def modInverse(a, n):
    """Calculate the multiplicative inverse of a (mod n)."""
    if a < n:
        # extended_gcd(n, a) returns n*x + a*y = 1.
        d, x, y = extended_gcd(n, a)
        inverse = y
    else:
        # extended_gcd(a, n) returns a*x + n*y = 1.
        d, x, y = extended_gcd(a, n)
        inverse = x

    if d != 1:
        raise ValueError("The modular inverse does not exist.")
    return inverse % n

def divide(b, a, n):
    assert n > 1 and a > 0 # and gcd(a, n) == 1

    aInv = modInverse(a,n)

    assert (a * aInv) % n == 1

    # return the number x s.t. a*x = b (mod n) and 0 <= x <= n-1.
    return (aInv * b) % n

In [11]:
print(divide(9, 5, 3))
print(divide(223, 67, 54))

0
13


5. Implement the function ***FastModularExponentiation(b, e, m)*** which computes $b^e \bmod{m}$ using  around $2\log_2{e}$ modular multiplications. You are not allowed to use Python built-in exponentiation functions. *(3 points)*

In [12]:
def FastModularExponentiation(b, e, m):
    if e < 0:
        raise ValueError("The exponent must be non-negative.")
    if m <= 0:
        raise ValueError("The modulus must be positive.")

    # Repeated squaring (binary exponentiation).
    result = 1 % m
    base = b % m
    exponent = e

    while exponent > 0:
        if exponent % 2 == 1:
            result = (result * base) % m
        base = (base * base) % m
        exponent //= 2

    return result

In [13]:
print(FastModularExponentiation(4, 5, 3))

1


6. Implement the algorithm to construct the number from the Chinese Remainder Theorem. *(1 point)*

You need to implement the function *ChineseRemainderTheorem( $ n_1, r_1, n_2, r_2 $ )* which takes two distinct coprime numbers $n_1$ and $n_2$ $( n_1 > n_2 )$ and the respective remainders $0 \leq r_1 < n_1$ and $0 \leq r_2  < n_2$, and returns the number $r$ such that $0 \leq r < n_1n_2$, $r \equiv r_1 \bmod{n_1}$ and $r \equiv r_2 \bmod{n_2}$.

In [14]:
def ChineseRemainderTheorem(n1, r1, n2, r2):
    # if you have implemented extended_gcd correctly, x and y should be two integers such that
    # n1*x + n2*y = gcd(n1, n2). Now use these two integers to return a proper answer.
    # make sure that you have run the extended_gcd function before running this cell
    (d, x, y) = extended_gcd(n1, n2)

    assert d == 1

    # n1*x + n2*y = 1, so n2*y is the inverse of n2 modulo n1,
    # and n1*x is the inverse of n1 modulo n2.
    r = (r1 * n2 * y + r2 * n1 * x) % (n1 * n2)
    return r

In [15]:
print(ChineseRemainderTheorem(13, 7, 6, 5)) # 59
print(ChineseRemainderTheorem(28, 20, 19, 3)) # 440

59
440
